# Heft 4: Den Zwilling fragen

Der Zwilling der Wasserstoffkette rechnet eine Nacht in Minutenschritten. Dieses Heft stellt ihm Fragen: erst eine,
dann dieselbe über viele Uhrzeiten. Die Eingänge kommen aus den Reihen der Signale des Systems (Wind der Station,
Preis), soweit es sie gibt; jede Antwort sagt, woher ihre Eingänge stammen und wie weit sie trägt.

Ein Lauf rechnet auf dem Server und wird dort im Speicher gehalten. In den Broker schreibt dieses Heft nichts.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from opentsx.client import RetrievalClient

RETRIEVAL = os.environ.get("OPENTSX_RETRIEVAL", "http://opentsx-retrieval.opentsx.svc:8080")
SYSTEM = "h2-chain"
client = RetrievalClient(RETRIEVAL)
print("Retrieval-Server:", RETRIEVAL)

## 1. Was sich fragen lässt

Das Szenario nennt seine Stellgrößen, die Fahrweisen des Elektrolyseurs und die Entscheidungen seiner Nacht.

In [ ]:
sc = client.scenario(SYSTEM)
print(sc["title"])
print(sc["decision"] or sc["description"])
print("Beginn", sc["start"], f"| {sc['steps']} Schritte zu {sc['stepSeconds']} s")
print("\nEntscheidungen der Nacht:")
for d in sc["referenceDecisions"]:
    print(f"  {d['clock'][:5]}  {d['atom']}.{d['control']} = {d['value']}  ({d['reason']})")
pd.DataFrame(sc["controls"])[["atom", "control", "default", "unit"]]

## 2. Eine Frage

Was wäre, wenn die Übergabestation ihr Ventil um 02:00 wieder öffnet? Gerechnet auf den Reihen der Signale und
verglichen mit den Entscheidungen der Nacht auf denselben Eingängen.

In [ ]:
run = client.run(SYSTEM, [{"at": "02:00", "atom": "handover", "control": "valve", "value": 1}], inputs="signals")
print("Lauf", run["run"], "|", run["label"])
print("verglichen mit", run["baseline"])
print("Status:", run["status"]["usability"])
for r in run["status"]["reasons"]:
    print("  -", r)
pd.DataFrame(run["inputs"])[["port", "source", "origin", "coverage"]]

In [ ]:
cmp = client.compare(SYSTEM, run["run"], run["baseline"])
kpis = pd.DataFrame(cmp["kpis"])[["label", "base", "run", "delta", "unit", "verdict"]]
kpis.columns = ["Kennzahl", "Referenz", "dieser Lauf", "Differenz", "Einheit", "Urteil"]
kpis.round(1)

In [ ]:
start = pd.Timestamp(cmp["start"])
step = pd.Timedelta(seconds=cmp["stepSeconds"])
fig, axes = plt.subplots(1, len(cmp["series"]), figsize=(15, 3.5))
for ax, (key, s) in zip(np.atleast_1d(axes), cmp["series"].items()):
    idx = pd.date_range(start, periods=len(s["base"]), freq=step)
    ax.plot(idx, [np.nan if v is None else v for v in s["base"]], color="#8a949e", label="Referenz")
    ax.plot(idx, [np.nan if v is None else v for v in s["run"]], color="#1f6fb3", label="dieser Lauf")
    for lim in s["limits"]:
        ax.axhline(lim["value"], color="#b3392b", ls="--", lw=0.8)
    ax.set_title(s["label"])
    ax.tick_params(axis="x", labelrotation=30)
np.atleast_1d(axes)[0].legend()
plt.tight_layout()
plt.show()

## 3. Dieselbe Frage über viele Uhrzeiten

Bis wann muss das Ventil wieder offen sein? Ein Lauf je Uhrzeit, von 01:10 bis 03:00 in Schritten von zehn Minuten.
Jede Zeile trägt die Kennung ihres Laufs.

In [ ]:
rows = []
for minute in range(70, 181, 10):
    at = f"{minute // 60:02d}:{minute % 60:02d}"
    r = client.run(SYSTEM, [{"at": at, "atom": "handover", "control": "valve", "value": 1}], inputs="signals")
    k = r["kpis"]
    rows.append({"Ventil offen um": at, "Schutzabschaltungen": k.get("ely.trips"), "höchster Druck im Puffer [bar]": k.get("buffer.p_max_bar"),
                 "geliefert [kg]": k.get("handover.delivered_kg"), "nicht geliefert [kg]": k.get("handover.shortfall_kg"), "Lauf": r["run"]})
sweep = pd.DataFrame(rows)
sweep.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(sweep["Ventil offen um"], sweep["höchster Druck im Puffer [bar]"], marker="o", label="höchster Druck im Puffer")
ax.axhline(200, color="#b3392b", ls="--", lw=0.8, label="Abschaltschwelle")
ax.set_ylabel("bar")
ax.set_xlabel("Ventil wieder offen um")
ax2 = ax.twinx()
ax2.bar(sweep["Ventil offen um"], sweep["nicht geliefert [kg]"], alpha=0.25, color="#c98a00", label="nicht geliefert")
ax2.set_ylabel("kg")
ax.legend(loc="upper left")
ax.set_title("Bis wann muss das Ventil offen sein?")
plt.show()
latest = sweep[sweep["Schutzabschaltungen"] == 0]["Ventil offen um"]
print("Ohne Schutzabschaltung bis:", latest.iloc[-1] if len(latest) else "keine der gerechneten Uhrzeiten")

## Was diese Antworten tragen

Jeder Lauf heißt `restricted`, und die Gründe stehen in seinem Status: Der Bedarf des Abnehmers ist ein
Demo-Generator, Fahrplan und Netzzähler fehlen, die Anlage ist angenommen. Die Rechnung zeigt den Mechanismus der
Kette, keine Wirkung, auf die ein Betrieb sich verlassen könnte. Mit einem Abnahmeprofil des Betreibers und seinen
Anlagendaten ändert sich das.